# 02 · Tracking data
Broadcast tracking at 10 frames per second. Players off camera are **extrapolated**
(`is_detected = False`), so it's worth knowing how much of the data is estimated, and when.

In [ ]:
import sys
from pathlib import Path

# Make the repo's src/ importable no matter where the notebook is run from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
from src.loading import match_table, load_tracking, load_match_meta, orient, FPS
from src.pitch import plot_frame

matches = match_table()
MATCH_ID = int(matches.match_id.iloc[0])     # change to explore another match
meta = load_match_meta(MATCH_ID)
print(f"{MATCH_ID}: {meta['home_team']} {meta['home_score']}-{meta['away_score']} {meta['away_team']}")

# step=5 keeps 2 frames per second, which is plenty for exploring and much faster to load
trk = load_tracking(MATCH_ID, step=5)
print(f"{trk.frame.nunique():,} frames, {len(trk):,} rows")
trk.head()

## How much is extrapolated?

In [ ]:
people = trk[trk.role != "ball"]
ball = trk[trk.role == "ball"]
print(f"player positions detected on camera: {people.is_detected.mean():.1%}")
print(f"frames with a ball position:         {trk.groupby('frame').role.apply(lambda r: (r == 'ball').any()).mean():.1%}")

people.groupby("role").is_detected.mean().sort_values().rename("share_detected").to_frame().round(3)

In [ ]:
# Detection over the match: broadcast cuts and close-ups show up as dips
by_min = people.assign(minute=(people.frame / FPS // 60).astype(int)).groupby(["period", "minute"]).is_detected.mean()
fig, ax = plt.subplots(figsize=(10, 3))
for per, s in by_min.groupby(level=0):
    ax.plot(s.index.get_level_values(1), s.values, label=f"period {per}")
ax.set_ylabel("share detected"); ax.set_xlabel("frame minute"); ax.set_ylim(0, 1); ax.legend()

In [ ]:
# Detection vs. distance from the ball: players far from the ball are more often off camera
f_ball = ball[["frame", "x", "y"]].rename(columns={"x": "bx", "y": "by"})
d = people.merge(f_ball, on="frame")
d["dist_to_ball"] = np.hypot(d.x - d.bx, d.y - d.by)
d.groupby(pd.cut(d.dist_to_ball, [0, 10, 20, 30, 40, 60, 120]), observed=True).is_detected.mean().round(3)

## Look at a frame
Hollow markers are extrapolated players.

In [ ]:
frame_no = int(trk.frame.quantile(0.3))
f = trk[trk.frame == frame_no]
plot_frame(f, length=meta["pitch_length"], width=meta["pitch_width"], color_by="team",
           title=f"Frame {frame_no}, period {int(f.period.iloc[0])}")
plt.show()

## Average positions
Oriented so each team attacks toward +x, so both halves can be combined.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
from src.pitch import draw_pitch
for ax, side in zip(axes, ["home", "away"]):
    team_id = meta[f"{side}_team_id"]
    o = orient(trk[trk.team_id == team_id], MATCH_ID, team_id)
    avg = o[~o.is_gk.fillna(False).astype(bool)].groupby(["player_id", "short_name"])[["x", "y"]].mean().reset_index()
    draw_pitch(ax, meta["pitch_length"], meta["pitch_width"])
    ax.scatter(avg.x, avg.y, s=80)
    for r in avg.itertuples():
        ax.annotate(r.short_name, (r.x, r.y), fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.set_title(f"{meta[side + '_team']} (attacking →)")
plt.tight_layout()